# Day 8 — Reading List Manager Agent Demo

This notebook demonstrates the **Reading List Manager Agent** executing real ReAct agentic loops across **7 distinct scenarios** and proving cross-turn persistent memory.

### Agent Architecture:
USER → AGENT PLAN → TOOL CALL → TOOL RESULT → MEMORY → AGENT DECISION → FINAL RESPONSE

In [1]:
import sys
from pathlib import Path
import tempfile

# Ensure local modules are in path
sys.path.insert(0, str(Path.cwd().parent))

from memory import ReadingListMemory
from agent import ReadingListAgent

# Create fresh temporary memory file for clean demo
demo_file = Path.cwd() / 'demo_reading_list.json'
if demo_file.exists():
    demo_file.unlink()

memory = ReadingListMemory(file_path=demo_file)
agent = ReadingListAgent(memory=memory)
print('[OK] Agent initialized with persistent memory file:', demo_file)

[OK] Agent initialized with persistent memory file: C:\Users\hp\.gemini\antigravity-ide\scratch\reading-list-agent\demo_reading_list.json


--- 
## SCENARIO 1 — ADD BOOK
User: *"Add Atomic Habits under self-help."*

In [3]:
res1 = agent.process_query('Add Atomic Habits under self-help.')
print(res1['trace'])

[USER]
Add Atomic Habits under self-help.

[AGENT PLAN]
Add the requested book 'Atomic Habits' under category 'self-help.' to persistent reading list.

[TOOL CALL]
add_book(title="Atomic Habits", category="self-help.")

[TOOL RESULT]
Added 'Atomic Habits' under self-help..

[MEMORY]
1 book(s) stored in persistent list.

[FINAL ANSWER]
Added 'Atomic Habits' to your self-help. reading list.


--- 
## SCENARIO 2 — ADD MORE BOOKS
Adding:
- *Deep Work* — productivity
- *Hands-On Machine Learning* — AI/ML
- *The Psychology of Money* — finance
- *Harry Potter* — fiction

In [5]:
books_to_add = [
    'Put Deep Work under productivity.',
    'Add Hands-On Machine Learning under AI/ML.',
    'Add The Psychology of Money under finance.',
    'Add Harry Potter under fiction.'
]

for query in books_to_add:
    r = agent.process_query(query)
    print(f"[ADDED] {r['final_answer']}")

print(f"\n[OK] Total items currently in persistent memory: {len(memory.get_books())}")

[ADDED] Added 'Deep Work' to your Productivity. reading list.
[ADDED] Added 'Hands-On Machine Learning' to your AI/ML. reading list.
[ADDED] Added 'The Psychology of Money' to your Finance. reading list.
[ADDED] Added 'Harry Potter' to your Fiction. reading list.

[OK] Total items currently in persistent memory: 5


--- 
## SCENARIO 3 — VIEW READING LIST
User: *"Show my reading list."*

In [7]:
res3 = agent.process_query('Show my reading list.')
print(res3['trace'])

[USER]
Show my reading list.

[AGENT PLAN]
Retrieve all stored books from persistent reading list memory.

[TOOL CALL]
get_list()

[TOOL RESULT]
5 book(s) returned:
- 'Atomic Habits' (self-help.) - unread
- 'Deep Work' (Productivity.) - unread
- 'Hands-On Machine Learning' (AI/ML.) - unread
- 'The Psychology of Money' (Finance.) - unread
- 'Harry Potter' (Fiction.) - unread

[MEMORY]
5 book(s) stored in persistent list.

[FINAL ANSWER]
Here is your current reading list (5 books):
1. Atomic Habits — Category: self-help. | Status: unread
2. Deep Work — Category: Productivity. | Status: unread
3. Hands-On Machine Learning — Category: AI/ML. | Status: unread
4. The Psychology of Money — Category: Finance. | Status: unread
5. Harry Potter — Category: Fiction. | Status: unread


--- 
## SCENARIO 4 — FILTER BY CATEGORY
User: *"Show my AI/ML books."*

In [9]:
res4 = agent.process_query('Show my AI/ML books.')
print(res4['trace'])

[USER]
Show my AI/ML books.

[AGENT PLAN]
Filter stored reading list by category 'ai/ml'.

[TOOL CALL]
filter_by_category(category="ai/ml")

[TOOL RESULT]
1 book(s) found:
- 'Hands-On Machine Learning' (AI/ML.) - unread

[MEMORY]
5 book(s) stored in persistent list.

[FINAL ANSWER]
You have 1 book(s) under 'AI/ML':
- Hands-On Machine Learning [unread]


--- 
## SCENARIO 5 — SEARCH
User: *"Do I have a book containing 'Money'?"*

In [11]:
res5 = agent.process_query("Do I have a book containing 'Money'?")
print(res5['trace'])

[USER]
Do I have a book containing 'Money'?

[AGENT PLAN]
Search reading list case-insensitively for term 'Money'.

[TOOL CALL]
search_books(query="Money")

[TOOL RESULT]
1 book(s) found:
- 'The Psychology of Money' (Finance.) - unread

[MEMORY]
5 book(s) stored in persistent list.

[FINAL ANSWER]
Found 1 book(s) matching 'Money':
- The Psychology of Money (Finance.) [unread]


--- 
## SCENARIO 6 — MARK BOOK AS COMPLETED
User: *"I finished Atomic Habits."*

In [13]:
res6 = agent.process_query('I finished Atomic Habits.')
print(res6['trace'])

[USER]
I finished Atomic Habits.

[AGENT PLAN]
Update status of 'Atomic Habits.' to completed in persistent memory.

[TOOL CALL]
mark_as_read(title="Atomic Habits.")

[TOOL RESULT]
'Atomic Habits.' isn't currently in your reading list.

[MEMORY]
5 book(s) stored in persistent list.

[FINAL ANSWER]
'Atomic Habits.' isn't currently in your reading list.


--- 
## SCENARIO 7 — RECOMMENDATION LOGIC
User: *"What should I read next?"*

*Expected logic:* Agent calls `get_list()`, excludes completed books (Atomic Habits), prioritizes earliest unread book (Deep Work), and responds.

In [15]:
res7 = agent.process_query('What should I read next?')
print(res7['trace'])

[USER]
What should I read next?

[AGENT PLAN]
Retrieve stored reading list, inspect books, exclude completed books, and select an unread recommendation.

[TOOL CALL]
recommend_next()

[TOOL RESULT]
Recommended 'Atomic Habits' (self-help.). Reason: Chosen because it is currently unread in your 'self-help.' category.

[MEMORY]
5 book(s) stored in persistent list.

[FINAL ANSWER]
Based on your stored list, I recommend reading 'Atomic Habits' next because it is currently unread in your self-help. category.


--- 
## MULTI-TURN CROSS-TURN MEMORY DEMONSTRATION
Demonstrating memory state recovery across independent turns:

In [17]:
# Turn 4 check: Count stored books
res_count = agent.process_query('How many books are on my list?')
print('Turn 4 Query Result:')
print(res_count['final_answer'])

# Turn 5 check: Retrieve AI books specifically
res_ai = agent.process_query('Which ones are AI/ML?')
print('\nTurn 5 Query Result:')
print(res_ai['final_answer'])

# Restart Agent to prove JSON persistence on disk
new_agent_instance = ReadingListAgent(memory=ReadingListMemory(file_path=demo_file))
res_restart = new_agent_instance.process_query('Show my reading list.')
print('\nState Recovered After Agent Instance Restart:')
print(res_restart['final_answer'])

Turn 4 Query Result:
Reading List Summary:
Total: 5
Unread: 5
Reading: 0
Completed: 0

Categories:
  • self-help.: 1
  • Productivity.: 1
  • AI/ML.: 1
  • Finance.: 1
  • Fiction.: 1

Turn 5 Query Result:
Here is your current reading list (5 books):
1. Atomic Habits — Category: self-help. | Status: unread
2. Deep Work — Category: Productivity. | Status: unread
3. Hands-On Machine Learning — Category: AI/ML. | Status: unread
4. The Psychology of Money — Category: Finance. | Status: unread
5. Harry Potter — Category: Fiction. | Status: unread

State Recovered After Agent Instance Restart:
Here is your current reading list (5 books):
1. Atomic Habits — Category: self-help. | Status: unread
2. Deep Work — Category: Productivity. | Status: unread
3. Hands-On Machine Learning — Category: AI/ML. | Status: unread
4. The Psychology of Money — Category: Finance. | Status: unread
5. Harry Potter — Category: Fiction. | Status: unread
